### Регрессия: стоимость билетов на самолет

Цель работы — построить модели регрессии для прогнозирования стоимости авиабилета на основе характеристик перелёта.

### Подключение модулей

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

sns.set_theme(style="whitegrid")
np.random.seed(42)

### Формирование набора данных

Для ноутбука используется самодостаточный учебный набор данных о перелётах. Данные содержат числовые и категориальные признаки, влияющие на стоимость билета.

In [ ]:
n = 5000

airlines = ['Aeroflot', 'S7 Airlines', 'Pobeda', 'Rossiya', 'Ural Airlines']
classes = ['Economy', 'Business']
stops = ['Без пересадок', '1 пересадка', '2 пересадки']
origins = ['Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 'Казань']
destinations = ['Сочи', 'Москва', 'Калининград', 'Владивосток', 'Иркутск']

df = pd.DataFrame({
    'Airline': np.random.choice(airlines, n),
    'Class': np.random.choice(classes, n, p=[0.85, 0.15]),
    'Stops': np.random.choice(stops, n, p=[0.60, 0.32, 0.08]),
    'Origin': np.random.choice(origins, n),
    'Destination': np.random.choice(destinations, n),
    'Distance_km': np.random.randint(400, 7500, n),
    'Duration_hours': np.round(np.random.uniform(1.0, 12.0, n), 1),
    'Days_before_departure': np.random.randint(1, 91, n),
    'Baggage_kg': np.random.choice([0, 10, 15, 20, 23, 32], n, p=[.08,.12,.20,.30,.18,.12])
})

airline_effect = df['Airline'].map({
    'Aeroflot': 900, 'S7 Airlines': 700, 'Pobeda': -500,
    'Rossiya': 300, 'Ural Airlines': 200
})
class_effect = df['Class'].map({'Economy': 0, 'Business': 12000})
stops_effect = df['Stops'].map({'Без пересадок': 2500, '1 пересадка': 0, '2 пересадки': -1800})
early_effect = (90 - df['Days_before_departure']) * 55
baggage_effect = df['Baggage_kg'] * 85
noise = np.random.normal(0, 1800, n)

df['Price'] = (
    2500 + df['Distance_km'] * 3.2 + df['Duration_hours'] * 650
    + airline_effect + class_effect + stops_effect
    + early_effect + baggage_effect + noise
).clip(1500, None).round(0)

df.head()

### Описание данных

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
print("Размер набора данных:", df.shape)
print("Пропущенные значения:")
print(df.isnull().sum())

### Первичный анализ данных

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Price'], kde=True)
plt.title('Распределение стоимости авиабилетов')
plt.xlabel('Стоимость, руб.')
plt.ylabel('Количество')
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Class', y='Price')
plt.title('Стоимость билетов в зависимости от класса')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='Distance_km', y='Price', hue='Class', alpha=0.5)
plt.title('Зависимость стоимости от расстояния')
plt.xlabel('Расстояние, км')
plt.ylabel('Стоимость, руб.')
plt.show()

In [ ]:
numeric_cols = ['Distance_km', 'Duration_hours', 'Days_before_departure', 'Baggage_kg', 'Price']
plt.figure(figsize=(9, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f')
plt.title('Корреляционная матрица')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='Airline', y='Price', errorbar=None)
plt.title('Средняя стоимость билета по авиакомпаниям')
plt.xticks(rotation=20)
plt.show()

### Подготовка данных для регрессии

In [ ]:
X = df.drop(columns='Price')
y = df['Price']

categorical_features = ['Airline', 'Class', 'Stops', 'Origin', 'Destination']
numeric_features = ['Distance_km', 'Duration_hours', 'Days_before_departure', 'Baggage_kg']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

### Линейная регрессия

In [ ]:
linear_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

print("Линейная регрессия обучена.")

### Метрики оценки качества регрессии

MAE — средняя абсолютная ошибка. MSE — средняя квадратичная ошибка. RMSE — корень из MSE. R² показывает долю объяснённой моделью вариации целевого признака.

In [ ]:
def evaluate(true, predicted):
    mae = metrics.mean_absolute_error(true, predicted)
    mse = metrics.mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2 = metrics.r2_score(true, predicted)
    return mae, mse, rmse, r2

def print_metrics(true, predicted):
    mae, mse, rmse, r2 = evaluate(true, predicted)
    print(f'MAE:  {mae:,.2f}')
    print(f'MSE:  {mse:,.2f}')
    print(f'RMSE: {rmse:,.2f}')
    print(f'R2:   {r2:.4f}')

print_metrics(y_test, linear_pred)

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=linear_pred, alpha=0.5)
plt.xlabel('Фактическая стоимость')
plt.ylabel('Предсказанная стоимость')
plt.title('Линейная регрессия: факт и прогноз')
plt.show()

### KNeighborsRegressor

In [ ]:
knn_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor(n_neighbors=10))
])
knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)
print_metrics(y_test, knn_pred)

### DecisionTreeRegressor

In [ ]:
tree_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=10, random_state=42))
])
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
print_metrics(y_test, tree_pred)

### RandomForestRegressor

In [ ]:
forest_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])
forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)
print_metrics(y_test, forest_pred)

### Сравнение моделей

In [ ]:
results = pd.DataFrame([
    ['Linear Regression', *evaluate(y_test, linear_pred)],
    ['KNeighborsRegressor', *evaluate(y_test, knn_pred)],
    ['DecisionTreeRegressor', *evaluate(y_test, tree_pred)],
    ['RandomForestRegressor', *evaluate(y_test, forest_pred)]
], columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2'])

results.sort_values('R2', ascending=False).reset_index(drop=True)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results, x='Model', y='R2')
plt.title('Сравнение моделей по R²')
plt.xticks(rotation=20)
plt.ylim(0, 1)
plt.show()

### Пример прогнозирования стоимости нового билета

In [ ]:
new_flight = pd.DataFrame([{
    'Airline': 'Aeroflot',
    'Class': 'Economy',
    'Stops': 'Без пересадок',
    'Origin': 'Москва',
    'Destination': 'Сочи',
    'Distance_km': 1600,
    'Duration_hours': 2.5,
    'Days_before_departure': 30,
    'Baggage_kg': 20
}])

prediction = forest_model.predict(new_flight)[0]
print(f'Прогнозируемая стоимость билета: {prediction:,.0f} руб.')

### Вывод

В работе была построена задача регрессии для прогнозирования стоимости авиабилетов. Были выполнены подготовка данных, исследовательский анализ, обучение нескольких регрессионных моделей и сравнение их по MAE, MSE, RMSE и R². Полученную модель можно использовать для оценки ориентировочной стоимости нового перелёта по его характеристикам.